In [16]:
import io
import boto3
import pandas as pd
import numpy as np
import sagemaker.amazon.common as smac

bucket = "vector-industries-ads508"
s3 = boto3.client("s3")

# Used gen AI to solve issue of csv compatability with factorization model.
# Gen AI code below.
train_df = pd.read_csv("train.csv", header=None)
val_df = pd.read_csv("validation.csv", header=None)

# First column = label, remaining columns = features
y_train = train_df.iloc[:, 0].astype("float32").to_numpy()
X_train = train_df.iloc[:, 1:].astype("float32").to_numpy()

y_val = val_df.iloc[:, 0].astype("float32").to_numpy()
X_val = val_df.iloc[:, 1:].astype("float32").to_numpy()

# Convert to RecordIO-protobuf in memory
buf_train = io.BytesIO()
smac.write_numpy_to_dense_tensor(buf_train, X_train, y_train)
buf_train.seek(0)

buf_val = io.BytesIO()
smac.write_numpy_to_dense_tensor(buf_val, X_val, y_val)
buf_val.seek(0)

# Upload converted files to S3
train_recordio_key = "training/train.recordio-protobuf"
val_recordio_key = "training/validation.recordio-protobuf"
# Gen AI code above.

s3.upload_fileobj(buf_train, bucket, train_recordio_key)
s3.upload_fileobj(buf_val, bucket, val_recordio_key)

print(f"Uploaded train to s3://{bucket}/{train_recordio_key}")
print(f"Uploaded validation to s3://{bucket}/{val_recordio_key}")

Uploaded train to s3://vector-industries-ads508/training/train.recordio-protobuf
Uploaded validation to s3://vector-industries-ads508/training/validation.recordio-protobuf


In [10]:
import pandas as pd

df = pd.read_csv("dataprep/netflix_combined_ratings.csv")

df_model = df[["rating", "user_id", "movie_id"]].copy()

df_model = df_model.dropna()
df_model["rating"] = df_model["rating"].astype(float)
df_model["user_id"] = df_model["user_id"].astype(int)
df_model["movie_id"] = df_model["movie_id"].astype(int)

In [11]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df_model, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

In [12]:
train_df.to_csv("train.csv", index=False, header=False)
val_df.to_csv("validation.csv", index=False, header=False)

In [13]:
import boto3

s3 = boto3.client("s3")
bucket = "vector-industries-ads508"

s3.upload_file("train.csv", bucket, "training/train.csv")
s3.upload_file("validation.csv", bucket, "training/validation.csv")

In [15]:
import sagemaker
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sess.boto_region_name
bucket = "vector-industries-ads508"

fm_image = sagemaker.image_uris.retrieve(
    framework="factorization-machines",
    region=region
)

fm = Estimator(
    image_uri=fm_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/models/",
    sagemaker_session=sess
)

fm.set_hyperparameters(
    predictor_type="regressor",
    feature_dim=2,
    epochs=20,
    num_factors=20,
    mini_batch_size=1000
)

train_input = TrainingInput(
    s3_data=f"s3://{bucket}/training/train.recordio-protobuf",
    content_type="application/x-recordio-protobuf"
)

validation_input = TrainingInput(
    s3_data=f"s3://{bucket}/training/validation.recordio-protobuf",
    content_type="application/x-recordio-protobuf"
)

fm.fit({
    "train": f"s3://{bucket}/training/train.recordio-protobuf",
    "validation": f"s3://{bucket}/training/validation.recordio-protobuf"
})

INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: factorization-machines-2026-03-27-12-51-05-863


2026-03-27 12:51:07 Starting - Starting the training job...
2026-03-27 12:51:22 Starting - Preparing the instances for training...
2026-03-27 12:52:04 Downloading - Downloading the training image..............Docker entrypoint called with argument(s): train
Running default environment configuration script
/opt/amazon/lib/python3.8/site-packages/mxnet/model.py:97: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if num_device is 1 and 'dist' not in kvstore:
[03/27/2026 12:54:25 INFO 140653431105344] Reading default configuration from /opt/amazon/lib/python3.8/site-packages/algorithm/resources/default-conf.json: {'epochs': 1, 'mini_batch_size': '1000', 'use_bias': 'true', 'use_linear': 'true', 'bias_lr': '0.1', 'linear_lr': '0.001', 'factors_lr': '0.0001', 'bias_wd': '0.01', 'linear_wd': '0.001', 'factors_wd': '0.00001', 'bias_init_method': 'normal', 'bias_init_sigma': '0.01', 'linear_init_method': 'normal', 'linear_init_sigma': '0.01', 'factors_init_method': 'normal', 'factors_i